In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# 1. Define the Network Structure
# Edges: (Source) -> (Destination)
model = DiscreteBayesianNetwork([
    ('Age', 'BP'),
    ('BP', 'HeartDisease'),
    ('Cholesterol', 'HeartDisease')
])

# 2. Defining individual Conditional Probability Tables (CPTs)
# P(Age)
cpd_age = TabularCPD(variable='Age', variable_card=2, values=[[0.6], [0.4]])

# P(BP | Age)
cpd_bp = TabularCPD(variable='BP', variable_card=2, 
                   values=[[0.7, 0.3], [0.3, 0.7]],
                   evidence=['Age'], evidence_card=[2])

# P(Cholesterol)
cpd_chol = TabularCPD(variable='Cholesterol', variable_card=2, values=[[0.7], [0.3]])

# P(HeartDisease | BP, Cholesterol)
cpd_hd = TabularCPD(variable='HeartDisease', variable_card=2,
                   values=[[0.9, 0.6, 0.7, 0.1], 
                           [0.1, 0.4, 0.3, 0.9]],
                   evidence=['BP', 'Cholesterol'], evidence_card=[2, 2])

# 3. Associating the CPDs with the network structure
model.add_cpds(cpd_age, cpd_bp, cpd_chol, cpd_hd)

# 4. Validating the model
assert model.check_model()

# 5. Inference: Predicting Heart Disease given Evidence
infer = VariableElimination(model)
# Example: What is the probability of heart disease if BP is High (1)?
q = infer.query(variables=['HeartDisease'], evidence={'BP': 1})
print(q)

+-----------------+---------------------+
| HeartDisease    |   phi(HeartDisease) |
+=================+=====================+
| HeartDisease(0) |              0.5200 |
+-----------------+---------------------+
| HeartDisease(1) |              0.4800 |
+-----------------+---------------------+
